### Camadas Silver

Bem-vindo à etapa de transformação do nosso pipeline de dados! Nesta aula, vamos evoluir os dados brutos armazenados na Camada Bronze para as camadas **Silver** (limpeza, parsing de JSON e enriquecimento)

## O que vamos aprender nesta aula?

1. **Camada Silver:** Transformar colunas brutas, padronizar textos e extrair informações que estavam escondidas dentro do campo JSON (`payload_detalhes`).
2. **Camada Silver (Tabela de Dimensão):** Criar uma tabela de suporte (`silver_clientes`) para simular dados de interações dos clientes.


## prompt para o Genie Code do Databricks
Atue como um Engenheiro de Dados Sênior e Especialista em Databricks. 
Preciso que você escreva um script utilizando **PySpark** (para um notebook no Databricks) para criar e popular a tabela da **Camada Silver** (`workspace.silver.silver_atendimentos`) a partir da nossa tabela da Camada Bronze (`workspace.raw.bronze_atendimentos_alumax_prtc`).

### Contexto do Projeto:
- **Empresa:** AluMax (atendimento omnichannel: WhatsApp, chat, telefone e e-mail).
- **Tabela de Origem:** `workspace.raw.bronze_atendimentos_alumax_prtc`
- **Estrutura Atual:** Contém colunas básicas (id_interacao, cliente_id, canal, departamento, status, data_hora) e uma coluna de payload bruto em formato JSON string chamada `payload_detalhes`.

### Requisitos Técnicos Obrigatórios para a Camada Silver:
1. **Criação do Schema:** Garanta a criação do database/schema `workspace.silver` caso ele não exista (`CREATE DATABASE IF NOT EXISTS workspace.silver;`).
2. **Padronização de Tipos e Textos:** 
   - Converta a coluna `data_hora` para o tipo `TIMESTAMP`.
   - Padronize os campos de texto (`canal`, `status`) para letras minúsculas (`LOWER`) e `departamento` com iniciais maiúsculas (`INITCAP`).
3. **Explosão / Parsing do JSON (`payload_detalhes`):** 
   - Extraia os campos aninhados de dentro do JSON dependendo do canal de atendimento utilizando. Por exemplo: extrair `numero_origem` ou `tempo_resposta_bot_seg` para WhatsApp, `protocolo_anatel` ou `duracao_chamada_seg` para Telefone, etc., transformando-os em colunas dedicadas e tipadas na Silver.
   A coluna tem aspas duplas, segue um exemplo. "{""dominio_email"": ""outlook.com"", ""tamanho_corpo_bytes"": 1506, ""anexos_quantidade"": 3, ""tempo_primeira_resposta_horas"": 3.09}
4. **Metadados de Auditoria:** Preserve o carimbo de ingestão original da Bronze e adicione um novo carimbo indicando o momento do processamento na Silver (`current_timestamp()`) e a tabela deve ser particionada pela data_particao igual a tabela workspace.raw.bronze_atendimentos_alumax_prtc, configura o modo de sobrescrita dinâmica por partição.
5. **Ambiente:** O código deve ser limpo, comentado passo a passo e otimizado para execução em cluster Single-Node do Databricks.

Por favor, forneça o script em Spark completo e pronto para execução.

In [0]:
# ===================================================================
# CAMADA SILVER: TRANSFORMAÇÃO E ENRIQUECIMENTO DE DADOS (PySpark)
# ===================================================================
# Empresa: AluMax - Atendimento Omnichannel
# Objetivo: Criar tabela silver_atendimentos com parsing de JSON e 
#           padronização de dados a partir da camada Bronze usando PySpark
# ===================================================================

# PASSO 1: Criação do Schema Silver
# Garante que o database silver existe antes de criar as tabelas

spark.sql("""
    CREATE DATABASE IF NOT EXISTS workspace.silver
    COMMENT 'Schema da Camada Silver - Dados limpos e enriquecidos'
""")

print("✓ Schema 'workspace.silver' criado/verificado com sucesso!")

In [0]:
# ===================================================================
# PASSO 2: Leitura dos Dados da Camada Bronze
# ===================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, BooleanType, DoubleType, TimestampType

# Lê a tabela Bronze
df_bronze = spark.table("workspace.raw.bronze_atendimentos_alumax_prtc")

# Verifica a estrutura dos dados
print(f"Total de registros na Bronze: {df_bronze.count():,}")
print("\nSchema da tabela Bronze:")
df_bronze.printSchema()

# Visualiza amostra dos dados brutos
print("\nAmostra dos dados Bronze:")
display(df_bronze.limit(3))

In [0]:
# ===================================================================
# PASSO 3: Transformação e Parsing do JSON
# ===================================================================

# IMPORTANTE: O payload_detalhes vem com aspas extras e aspas internas duplicadas
# Exemplo: "{""browser"": ""Safari"", ...}"
# Precisamos limpar isso primeiro antes de fazer o parsing

# Criar coluna com payload limpo
df_bronze_clean = df_bronze.withColumn(
    "payload_limpo",
    F.regexp_replace(
        F.regexp_replace(
            F.col("payload_detalhes"),
            '^"|"\'$',  # Remove aspas do início e fim
            ''
        ),
        '""',  # Substitui aspas duplas por aspas simples
        '"'
    )
)

# Aplicar transformações básicas e parsing condicional do JSON
df_silver = df_bronze_clean.select(
    # ===================================================================
    # COLUNAS BÁSICAS (padronizadas)
    # ===================================================================
    F.col("id_interacao"),
    F.col("cliente_id"),
    
    # Padronização de texto: minúsculas para canal e status
    F.lower(F.col("canal")).alias("canal"),
    F.lower(F.col("status")).alias("status"),
    
    # Padronização de texto: iniciais maiúsculas para departamento
    F.initcap(F.col("departamento")).alias("departamento"),
    
    # Conversão explícita para TIMESTAMP
    F.col("data_hora").cast(TimestampType()).alias("data_hora"),
    
    F.col("data_particao").alias("data_particao"),
    # ===================================================================
    # PARSING JSON - CAMPOS ESPECÍFICOS DO WHATSAPP
    # ===================================================================
    F.when(
        F.lower(F.col("canal")) == "whatsapp",
        F.get_json_object(F.col("payload_limpo"), "$.numero_origem")
    ).alias("whatsapp_numero_origem"),
    
    F.when(
        F.lower(F.col("canal")) == "whatsapp",
        F.get_json_object(F.col("payload_limpo"), "$.atendente_bot").cast(BooleanType())
    ).alias("whatsapp_atendente_bot"),
    
    F.when(
        F.lower(F.col("canal")) == "whatsapp",
        F.get_json_object(F.col("payload_limpo"), "$.tempo_resposta_bot_seg").cast(IntegerType())
    ).alias("whatsapp_tempo_resposta_bot_seg"),
    
    F.when(
        F.lower(F.col("canal")) == "whatsapp",
        F.get_json_object(F.col("payload_limpo"), "$.versao_whatsapp_api")
    ).alias("whatsapp_versao_api"),
    
    # ===================================================================
    # PARSING JSON - CAMPOS ESPECÍFICOS DO TELEFONE
    # ===================================================================
    F.when(
        F.lower(F.col("canal")) == "telefone",
        F.get_json_object(F.col("payload_limpo"), "$.duracao_chamada_seg").cast(IntegerType())
    ).alias("telefone_duracao_chamada_seg"),
    
    F.when(
        F.lower(F.col("canal")) == "telefone",
        F.get_json_object(F.col("payload_limpo"), "$.fila_espera_seg").cast(IntegerType())
    ).alias("telefone_fila_espera_seg"),
    
    F.when(
        F.lower(F.col("canal")) == "telefone",
        F.get_json_object(F.col("payload_limpo"), "$.protocolo")
    ).alias("telefone_protocolo"),
    
    F.when(
        F.lower(F.col("canal")) == "telefone",
        F.get_json_object(F.col("payload_limpo"), "$.transferencias").cast(IntegerType())
    ).alias("telefone_transferencias"),
    
    # ===================================================================
    # PARSING JSON - CAMPOS ESPECÍFICOS DO EMAIL
    # ===================================================================
    F.when(
        F.lower(F.col("canal")) == "email",
        F.get_json_object(F.col("payload_limpo"), "$.dominio_email")
    ).alias("email_dominio"),
    
    F.when(
        F.lower(F.col("canal")) == "email",
        F.get_json_object(F.col("payload_limpo"), "$.tamanho_corpo_bytes").cast(IntegerType())
    ).alias("email_tamanho_corpo_bytes"),
    
    F.when(
        F.lower(F.col("canal")) == "email",
        F.get_json_object(F.col("payload_limpo"), "$.anexos_quantidade").cast(IntegerType())
    ).alias("email_anexos_quantidade"),
    
    F.when(
        F.lower(F.col("canal")) == "email",
        F.get_json_object(F.col("payload_limpo"), "$.tempo_primeira_resposta_horas").cast(DoubleType())
    ).alias("email_tempo_primeira_resposta_horas"),
    
    # ===================================================================
    # PARSING JSON - CAMPOS ESPECÍFICOS DO CHAT SITE
    # ===================================================================
    F.when(
        F.lower(F.col("canal")) == "chat_site",
        F.get_json_object(F.col("payload_limpo"), "$.browser")
    ).alias("chat_browser"),
    
    F.when(
        F.lower(F.col("canal")) == "chat_site",
        F.get_json_object(F.col("payload_limpo"), "$.ip_origem")
    ).alias("chat_ip_origem"),
    
    F.when(
        F.lower(F.col("canal")) == "chat_site",
        F.get_json_object(F.col("payload_limpo"), "$.pagina_origem")
    ).alias("chat_pagina_origem"),
    
    F.get_json_object(F.col("payload_limpo"), "$.satisfacao_pre_atendimento").cast(IntegerType()).alias("satisfacao_pre_atendimento"),
    
    # ===================================================================
    # METADADOS DE AUDITORIA
    # ===================================================================
    # Preserva o timestamp de ingestão original da Bronze
    F.col("_ingestion_timestamp").alias("bronze_ingestion_timestamp"),
    
    # Adiciona timestamp de processamento na Silver
    F.current_timestamp().alias("silver_processing_timestamp"),
    
    # Preserva informação do arquivo fonte
    F.col("_source_file")
)

print("✓ Transformações aplicadas com sucesso!")
print(f"\nTotal de colunas na Silver: {len(df_silver.columns)}")
print("\nSchema da tabela Silver:")
df_silver.printSchema()

In [0]:
df_silver.display()

In [0]:
# ===================================================================
# PASSO 4: Gravação da Tabela Silver
# ===================================================================

# Salva o DataFrame como tabela Delta no catálogo Unity Catalog
# Usa a opção partitionOverwriteMode diretamente no write para compatibilidade com Serverless
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("partitionOverwriteMode", "dynamic") \
    .partitionBy("data_particao") \
    .saveAsTable("workspace.silver.silver_atendimentos")

print("✓ Tabela 'workspace.silver.silver_atendimentos' criada com sucesso!")
print(f"\nTotal de registros gravados: {df_silver.count():,}")

In [0]:
# ===================================================================
# PASSO 5: Validação e Verificação dos Dados
# ===================================================================

# Lê a tabela Silver recém-criada para validação
df_validation = spark.table("workspace.silver.silver_atendimentos")

# Estatísticas gerais
total_registros = df_validation.count()
total_canais = df_validation.select("canal").distinct().count()
total_departamentos = df_validation.select("departamento").distinct().count()

print("=" * 60)
print("VALIDAÇÃO DA TABELA SILVER")
print("=" * 60)
print(f"Total de registros: {total_registros:,}")
print(f"Total de canais: {total_canais}")
print(f"Total de departamentos: {total_departamentos}")
print("=" * 60)

# Distribuição por canal
print("\nDistribuição de registros por canal:")
df_validation.groupBy("canal") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

# Amostra de dados por canal para validar parsing do JSON
print("\nExemplos de campos específicos por canal:")
df_validation.groupBy("canal").agg(
    F.count("*").alias("quantidade"),
    F.max(F.when(F.col("canal") == "whatsapp", F.col("whatsapp_numero_origem"))).alias("exemplo_whatsapp"),
    F.max(F.when(F.col("canal") == "telefone", F.col("telefone_protocolo"))).alias("exemplo_telefone"),
    F.max(F.when(F.col("canal") == "email", F.col("email_dominio"))).alias("exemplo_email"),
    F.max(F.when(F.col("canal") == "chat_site", F.col("chat_browser"))).alias("exemplo_chat")
).orderBy(F.desc("quantidade")).show(truncate=False)

# Visualiza amostra completa dos dados transformados
print("\nAmostra de 10 registros da tabela Silver:")
display(df_validation.limit(10))

In [0]:
%sql
select * from workspace.silver.silver_atendimentos

In [0]:
# ===================================================================
# ANÁLISES E INSIGHTS - ATENDIMENTOS OMNICHANNEL
# ===================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Carrega a tabela Silver
df = spark.table("workspace.silver.silver_atendimentos")

print("="*70)
print("ANÁLISE 1: VISÃO GERAL DE ATENDIMENTOS")
print("="*70)

# Métricas gerais
total_atendimentos = df.count()
total_clientes_unicos = df.select("cliente_id").distinct().count()

print(f"\n📊 Total de Atendimentos: {total_atendimentos:,}")
print(f"👥 Total de Clientes Únicos: {total_clientes_unicos:,}")
print(f"📈 Média de Atendimentos por Cliente: {total_atendimentos/total_clientes_unicos:.2f}")

# Distribuição por canal
print("\n🔹 Distribuição por Canal de Atendimento:")
df_canal = df.groupBy("canal").agg(
    F.count("*").alias("total_atendimentos"),
    F.countDistinct("cliente_id").alias("clientes_unicos")
).withColumn(
    "percentual",
    F.round((F.col("total_atendimentos") / total_atendimentos * 100), 2)
).orderBy(F.desc("total_atendimentos"))

display(df_canal)

In [0]:
print("="*70)
print("ANÁLISE 2: PERFORMANCE POR CANAL E STATUS")
print("="*70)

# Taxa de resolução por canal
print("\n🎯 Taxa de Resolução por Canal:")
df_status = df.groupBy("canal", "status").agg(
    F.count("*").alias("quantidade")
).withColumn(
    "total_canal",
    F.sum("quantidade").over(Window.partitionBy("canal"))
).withColumn(
    "percentual",
    F.round((F.col("quantidade") / F.col("total_canal") * 100), 2)
).orderBy("canal", F.desc("quantidade"))

display(df_status)

# Departamentos mais acionados
print("\n🏢 Top 5 Departamentos Mais Acionados:")
df_dept = df.groupBy("departamento", "canal").agg(
    F.count("*").alias("atendimentos")
).groupBy("departamento").agg(
    F.sum("atendimentos").alias("total_atendimentos"),
    F.count("canal").alias("canais_utilizados")
).orderBy(F.desc("total_atendimentos")).limit(5)

display(df_dept)

In [0]:
print("="*70)
print("ANÁLISE 3: MÉTRICAS DE TEMPO E EFICIÊNCIA POR CANAL")
print("="*70)

# Análise WhatsApp - Tempo de resposta do bot
print("\n📱 WhatsApp - Análise de Atendimento por Bot:")
df_whatsapp = df.filter(F.col("canal") == "whatsapp").select(
    F.avg("whatsapp_tempo_resposta_bot_seg").alias("tempo_medio_resposta_bot"),
    F.max("whatsapp_tempo_resposta_bot_seg").alias("tempo_maximo_resposta"),
    F.min("whatsapp_tempo_resposta_bot_seg").alias("tempo_minimo_resposta"),
    F.sum(F.when(F.col("whatsapp_atendente_bot") == True, 1).otherwise(0)).alias("atendidos_por_bot"),
    F.count("*").alias("total_whatsapp")
).withColumn(
    "percentual_bot",
    F.round((F.col("atendidos_por_bot") / F.col("total_whatsapp") * 100), 2)
)

display(df_whatsapp)

# Análise Telefone - Duração e tempo de espera
print("\n📞 Telefone - Análise de Duração e Fila:")
df_telefone = df.filter(F.col("canal") == "telefone").select(
    F.avg("telefone_duracao_chamada_seg").alias("duracao_media_chamada"),
    F.avg("telefone_fila_espera_seg").alias("tempo_medio_fila"),
    F.max("telefone_fila_espera_seg").alias("tempo_maximo_fila"),
    F.avg("telefone_transferencias").alias("media_transferencias"),
    F.sum(F.when(F.col("telefone_transferencias") > 0, 1).otherwise(0)).alias("chamadas_com_transferencia"),
    F.count("*").alias("total_telefone")
).withColumn(
    "percentual_transferencias",
    F.round((F.col("chamadas_com_transferencia") / F.col("total_telefone") * 100), 2)
)

display(df_telefone)

# Análise Email - Tempo de primeira resposta
print("\n📧 Email - Análise de Tempo de Resposta:")
df_email = df.filter(F.col("canal") == "email").select(
    F.avg("email_tempo_primeira_resposta_horas").alias("tempo_medio_primeira_resposta"),
    F.max("email_tempo_primeira_resposta_horas").alias("tempo_maximo_resposta"),
    F.min("email_tempo_primeira_resposta_horas").alias("tempo_minimo_resposta"),
    F.avg("email_anexos_quantidade").alias("media_anexos"),
    F.sum(F.when(F.col("email_anexos_quantidade") > 0, 1).otherwise(0)).alias("emails_com_anexos"),
    F.count("*").alias("total_emails")
).withColumn(
    "percentual_com_anexos",
    F.round((F.col("emails_com_anexos") / F.col("total_emails") * 100), 2)
)

display(df_email)

In [0]:
print("="*70)
print("ANÁLISE 4: TENDÊNCIAS TEMPORAIS")
print("="*70)

# Análise por hora do dia
print("\n⏰ Distribuição de Atendimentos por Hora do Dia:")
df_hora = df.withColumn(
    "hora_atendimento",
    F.hour("data_hora")
).groupBy("hora_atendimento", "canal").agg(
    F.count("*").alias("total_atendimentos")
).orderBy("hora_atendimento", "canal")

display(df_hora)

# Análise por dia da semana
print("\n📅 Distribuição de Atendimentos por Dia da Semana:")
df_dia_semana = df.withColumn(
    "dia_semana",
    F.date_format("data_hora", "EEEE")
).withColumn(
    "dia_semana_num",
    F.dayofweek("data_hora")
).groupBy("dia_semana_num", "dia_semana", "canal").agg(
    F.count("*").alias("total_atendimentos")
).orderBy("dia_semana_num", "canal")

display(df_dia_semana)

# Volume diário de atendimentos
print("\n📈 Evolução Diária de Atendimentos:")
df_diario = df.withColumn(
    "data",
    F.date_trunc("day", "data_hora")
).groupBy("data", "canal").agg(
    F.count("*").alias("total_atendimentos")
).orderBy("data", "canal")

display(df_diario)

In [0]:
print("="*70)
print("ANÁLISE 5: INSIGHTS AVANÇADOS E RECOMENDAÇÕES")
print("="*70)

# Clientes com mais atendimentos (possível problema recorrente)
print("\n⚠️ Top 10 Clientes com Mais Atendimentos:")
df_top_clientes = df.groupBy("cliente_id").agg(
    F.count("*").alias("total_atendimentos"),
    F.countDistinct("canal").alias("canais_utilizados"),
    F.countDistinct("departamento").alias("departamentos_acionados"),
    F.sum(F.when(F.col("status") == "resolvido", 1).otherwise(0)).alias("atendimentos_resolvidos")
).withColumn(
    "taxa_resolucao",
    F.round((F.col("atendimentos_resolvidos") / F.col("total_atendimentos") * 100), 2)
).orderBy(F.desc("total_atendimentos")).limit(10)

display(df_top_clientes)

# Chat Site - Análise de browsers e satisfação
print("\n💻 Chat Site - Análise de Browsers e Satisfação:")
df_chat = df.filter(F.col("canal") == "chat_site").groupBy("chat_browser").agg(
    F.count("*").alias("total_atendimentos"),
    F.avg("chat_satisfacao_pre_atendimento").alias("satisfacao_media")
).orderBy(F.desc("total_atendimentos"))

display(df_chat)

# Análise de domínios de email mais comuns
print("\n📨 Top 10 Domínios de Email Mais Comuns:")
df_dominios = df.filter(F.col("canal") == "email").groupBy("email_dominio").agg(
    F.count("*").alias("total_emails"),
    F.avg("email_tempo_primeira_resposta_horas").alias("tempo_medio_resposta")
).orderBy(F.desc("total_emails")).limit(10)

display(df_dominios)

print("\n" + "="*70)
print("✅ ANÁLISE CONCLUÍDA!")
print("="*70)